In [ ]:
import os
os.system("ls /kaggle/input/")

In [ ]:
os.system("ls /kaggle/input/competitions/")
os.system("ls /kaggle/input/datasets/")

os.system("ls /kaggle/input/datasets/ghostfreak135z/sherlock-test-features/kaggle/working/TestFeatures")
os.system("ls /kaggle/input/datasets/ghostfreak135z/sherlock-train-features/kaggle/working/TrainFeatures")



In [ ]:
TRAIN_FEAT_DIR = "/kaggle/input/datasets/ghostfreak135z/sherlock-train-features/kaggle/working/TrainFeatures"
TEST_FEAT_DIR  = "/kaggle/input/datasets/ghostfreak135z/sherlock-test-features/kaggle/working/TestFeatures"
LABEL_PATH     = "/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/train_labels.json"

In [ ]:
os.system("ls /kaggle/input/datasets/ghostfreak135z/sherlock-train-features/kaggle/working/TrainFeatures/ | head -5")

In [ ]:
mismatches = 0
for vid in labels:
    feat_path = os.path.join(TRAIN_FEAT_DIR, vid + ".pt")
    if not os.path.exists(feat_path):
        continue
    feats = torch.load(feat_path)
    if len(feats) != len(labels[vid]):
        mismatches += 1

print(f"Mismatched videos: {mismatches} / {len(labels)}")

In [ ]:
for vid in list(labels.keys())[:100]:
    feat_path = os.path.join(TRAIN_FEAT_DIR, vid + ".pt")
    if not os.path.exists(feat_path):
        continue
    feats = torch.load(feat_path)
    print(f"{vid}: feats={len(feats)}, label={len(labels[vid])}, diff={len(labels[vid])-len(feats)}")

# re extract frames

In [ ]:
import cv2
import os
from tqdm import tqdm

TRAIN_VIDEO_DIR = "/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/train"
TRAIN_FRAME_DIR = "/kaggle/working/Frames"

os.makedirs(TRAIN_FRAME_DIR, exist_ok=True)

video_files = sorted(os.listdir(TRAIN_VIDEO_DIR))
print(f"Total train videos: {len(video_files)}")

for vid_file in tqdm(video_files):
    if not vid_file.endswith(".mp4"):
        continue
        
    vid_id     = vid_file.replace(".mp4", "")
    out_folder = os.path.join(TRAIN_FRAME_DIR, vid_id)

    # Force re-extraction — delete existing folder if present
    if os.path.exists(out_folder):
        os.system(f"rm -rf {out_folder}")
    
    os.makedirs(out_folder, exist_ok=True)

    cap       = cv2.VideoCapture(os.path.join(TRAIN_VIDEO_DIR, vid_file))
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # Zero-padded — no sort issues ever
        cv2.imwrite(os.path.join(out_folder, f"{frame_idx:04d}.jpg"), frame)
        frame_idx += 1

    cap.release()

print("Done")

In [ ]:
import json

with open("/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/train_labels.json") as f:
    labels = json.load(f)

mismatches = 0
for vid_file in os.listdir(TRAIN_FRAME_DIR):
    vid_id = vid_file
    if vid_id not in labels:
        continue
    frame_count = len(os.listdir(os.path.join(TRAIN_FRAME_DIR, vid_id)))
    label_count = len(labels[vid_id])
    if frame_count != label_count:
        mismatches += 1
        
print(f"Mismatches: {mismatches}")

In [ ]:
import os
import torch
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm

DEVICE = "cuda"
TRAIN_FRAME_DIR = "/kaggle/working/Frames"
TRAIN_FEAT_DIR  = "/kaggle/working/TrainFeatures"

os.makedirs(TRAIN_FEAT_DIR, exist_ok=True)

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

print("Loading DINOv2...")
encoder = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(DEVICE)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False
print("DINOv2 loaded")

CHUNK = 64

videos = sorted(os.listdir(TRAIN_FRAME_DIR))
print(f"Total videos: {len(videos)}")

for vid in tqdm(videos):
    out_path = os.path.join(TRAIN_FEAT_DIR, vid + ".pt")
    
    # Force overwrite old features
    if os.path.exists(out_path):
        os.remove(out_path)

    folder = os.path.join(TRAIN_FRAME_DIR, vid)
    files  = sorted(os.listdir(folder),
                    key=lambda x: int(os.path.splitext(x)[0]))

    imgs = []
    for f in files:
        img = Image.open(os.path.join(folder, f)).convert("RGB")
        imgs.append(transform(img))

    if len(imgs) == 0:
        print(f"WARNING: {vid} has no frames, skipping")
        continue

    frames_tensor = torch.stack(imgs)

    feats = []
    with torch.no_grad():
        for start in range(0, len(frames_tensor), CHUNK):
            chunk = frames_tensor[start:start+CHUNK].to(DEVICE)
            feats.append(encoder(chunk).cpu())

    features = torch.cat(feats, dim=0)  # [N, 384]
    torch.save(features, out_path)

print("Done")

In [ ]:
import json, os, torch

with open("/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/train_labels.json") as f:
    labels = json.load(f)

mismatches = 0
for vid in labels:
    feat_path = os.path.join("/kaggle/working/TrainFeatures", vid + ".pt")
    if not os.path.exists(feat_path):
        continue
    feats = torch.load(feat_path)
    if len(feats) != len(labels[vid]):
        mismatches += 1
        print(f"{vid}: feats={len(feats)}, label={len(labels[vid])}")

print(f"\nTotal mismatches: {mismatches}")

In [ ]:
import os

print("Zipping TrainFeatures...")
os.system("zip -r /kaggle/working/TrainFeatures.zip /kaggle/working/TrainFeatures/")
os.system("ls -lh /kaggle/working/TrainFeatures.zip")

In [ ]:
# Update existing dataset with new version
metadata = """{
  "title": "sherlock-train-features",
  "id": "ghostfreak135z/sherlock-train-features",
  "licenses": [{"name": "CC0-1.0"}]
}"""

os.makedirs("/kaggle/working/train_upload", exist_ok=True)
with open("/kaggle/working/train_upload/dataset-metadata.json", "w") as f:
    f.write(metadata)

os.system("cp /kaggle/working/TrainFeatures.zip /kaggle/working/train_upload/")

print("Uploading...")
os.system("kaggle datasets version -p /kaggle/working/train_upload -m 'fixed frame extraction'")
print("Done!")

# # delete upload 

In [1]:
import os
os.system("rm -rf /kaggle/working/Frames")
os.system("rm -rf /kaggle/working/TestFrames")
os.system("rm -f /kaggle/working/TrainFeatures.zip")
os.system("rm -rf /kaggle/working/train_upload")
os.system("df -h /kaggle/working")

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   13G  7.2G  64% /kaggle/working


0

# # train

In [1]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.stats import kendalltau

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_FEAT_DIR = "/kaggle/input/datasets/ghostfreak135z/sherlock-train-features/kaggle/working/TrainFeatures"
TEST_FEAT_DIR  = "/kaggle/input/datasets/ghostfreak135z/sherlock-test-features/kaggle/working/TestFeatures"
LABEL_PATH     = "/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26/train_labels.json"

DEVICE  = "cuda"
EPOCHS  = 50
LR      = 1e-4

# ── Labels ────────────────────────────────────────────────────────────────────
with open(LABEL_PATH) as f:
    labels = json.load(f)

# ── Dataset ───────────────────────────────────────────────────────────────────
class FeatureDataset(Dataset):
    def __init__(self, feat_dir, labels):
        self.feat_dir = feat_dir
        self.labels   = labels
        self.videos   = [
            v for v in labels
            if os.path.exists(os.path.join(feat_dir, v + ".pt"))
            and len(torch.load(os.path.join(feat_dir, v + ".pt"))) == len(labels[v])
        ]
        print(f"Found {len(self.videos)} valid videos")

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        vid   = self.videos[idx]
        feats = torch.load(os.path.join(self.feat_dir, vid + ".pt"))  # [N, 384]
        label = self.labels[vid]   # [N] 1-indexed
        N     = len(feats)

        # Convert label to per-frame score target
        # label[pos] = frame_idx → score[frame_idx-1] = pos (normalized)
        scores = torch.zeros(N)
        for chron_pos, frame_idx in enumerate(label):
            scores[frame_idx - 1] = chron_pos / max(N - 1, 1)

        return feats, scores, N


def collate_fn(batch):
    feats  = [b[0] for b in batch]   # list of [N_i, 384]
    scores = [b[1] for b in batch]   # list of [N_i]
    lengths = [b[2] for b in batch]  # list of ints
    return feats, scores, lengths


# Train/val split
all_videos = list(labels.keys())
all_videos = [v for v in all_videos
              if os.path.exists(os.path.join(TRAIN_FEAT_DIR, v + ".pt"))]

np.random.seed(42)
np.random.shuffle(all_videos)
split      = int(0.9 * len(all_videos))
train_vids = {v: labels[v] for v in all_videos[:split]}
val_vids   = {v: labels[v] for v in all_videos[split:]}

print(f"Train: {len(train_vids)}, Val: {len(val_vids)}")

train_ds     = FeatureDataset(TRAIN_FEAT_DIR, train_vids)
val_ds       = FeatureDataset(TRAIN_FEAT_DIR, val_vids)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

# ── Model ─────────────────────────────────────────────────────────────────────
class TemporalTransformer(nn.Module):
    def __init__(self, d_model=384, nhead=8, num_layers=4, dim_ff=1024, dropout=0.1):
        super().__init__()
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head        = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

    def _sinusoidal(self, N, d_model, device):
        pe  = torch.zeros(N, d_model, device=device)
        pos = torch.arange(N, device=device).float().unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, device=device).float()
            * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe

    def forward(self, x):
        # x: [1, N, 384]
        N  = x.shape[1]
        pe = self._sinusoidal(N, x.shape[-1], x.device).unsqueeze(0)
        x  = x + pe
        x  = self.transformer(x)
        return self.head(x).squeeze(-1).squeeze(0)   # [N]


# ── Loss ──────────────────────────────────────────────────────────────────────
def ranking_loss(scores, target):
    # scores, target: [N]
    s_diff = scores.unsqueeze(1) - scores.unsqueeze(0)   # [N, N]
    t_diff = target.unsqueeze(1) - target.unsqueeze(0)   # [N, N]

    loss = torch.relu(1.0 - s_diff * t_diff.sign())

    # Upper triangle only, skip ties
    mask = torch.triu(torch.ones(len(scores), len(scores),
                                 device=scores.device), diagonal=1)
    mask = mask * (t_diff.abs() > 1e-6).float()

    denom = mask.sum()
    if denom < 1:
        return torch.tensor(0.0, device=scores.device, requires_grad=True)
    return (loss * mask).sum() / denom


# ── Validation ────────────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    taus = []
    with torch.no_grad():
        for feats_list, scores_list, lengths in loader:
            for feats, true_scores in zip(feats_list, scores_list):
                feats = feats.unsqueeze(0).to(DEVICE)   # [1, N, 384]
                pred  = model(feats).cpu().numpy()
                true  = true_scores.numpy()

                pred_order = np.argsort(pred)
                true_order = np.argsort(true)
                tau, _     = kendalltau(pred_order, true_order)
                taus.append(tau)
    model.train()
    return float(np.mean(taus))


# ── Training ──────────────────────────────────────────────────────────────────
model     = TemporalTransformer().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_tau = -1.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for feats_list, scores_list, lengths in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch_loss = torch.tensor(0.0, device=DEVICE)

        for feats, true_scores in zip(feats_list, scores_list):
            feats       = feats.unsqueeze(0).to(DEVICE)      # [1, N, 384]
            true_scores = true_scores.to(DEVICE)

            pred = model(feats)                               # [N]
            loss = ranking_loss(pred, true_scores)
            batch_loss = batch_loss + loss

        batch_loss = batch_loss / len(feats_list)
        optimizer.zero_grad()
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += batch_loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    val_tau  = evaluate(model, val_loader)
    print(f"Epoch {epoch+1:3d} | loss {avg_loss:.4f} | val τ {val_tau:.4f}")

    if val_tau > best_tau:
        best_tau = val_tau
        torch.save(model.state_dict(), "/kaggle/working/best_model.pt")
        print(f"  ✓ saved  best τ = {best_tau:.4f}")


# Zip model
os.system("zip /kaggle/working/model.zip /kaggle/working/best_model.pt")

# Upload model as dataset
metadata = """{
  "title": "sherlock-best-model",
  "id": "ghostfreak135z/sherlock-best-model",
  "licenses": [{"name": "CC0-1.0"}]
}"""

os.makedirs("/kaggle/working/model_upload", exist_ok=True)
with open("/kaggle/working/model_upload/dataset-metadata.json", "w") as f:
    f.write(metadata)

os.system("cp /kaggle/working/best_model.pt /kaggle/working/model_upload/")

print("Uploading model...")
os.system("kaggle datasets create -p /kaggle/working/model_upload")
print("Model uploaded!")

Train: 5049, Val: 562
Found 5047 valid videos
Found 562 valid videos


Epoch 1: 100%|██████████| 631/631 [00:53<00:00, 11.75it/s]


Epoch   1 | loss 0.9900 | val τ 0.0224
  ✓ saved  best τ = 0.0224


Epoch 2: 100%|██████████| 631/631 [00:53<00:00, 11.79it/s]


Epoch   2 | loss 0.9429 | val τ 0.0451
  ✓ saved  best τ = 0.0451


...

Epoch  49 | loss 0.2628 | val τ 0.0478


Epoch 50: 100%|██████████| 631/631 [00:52<00:00, 12.06it/s]


Epoch  50 | loss 0.2629 | val τ 0.0474
  adding: kaggle/working/best_model.pt (deflated 8%)
Uploading model...
Starting upload for file best_model.pt


100%|██████████| 21.3M/21.3M [00:00<00:00, 24.2MB/s]


Upload successful: best_model.pt (21MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/ghostfreak135z/sherlock-best-model
Model uploaded!


# # inference 

In [4]:
import os
import torch
import torch.nn as nn
import numpy as np
import csv

DEVICE        = "cuda"
TEST_FEAT_DIR = "/kaggle/input/datasets/ghostfreak135z/sherlock-test-features/kaggle/working/TestFeatures"
MODEL_PATH    = "/kaggle/input/datasets/ghostfreak135z/sherlock-best-model/best_model.pt"

# ── Model ─────────────────────────────────────────────────────────────────────
class TemporalTransformer(nn.Module):
    def __init__(self, d_model=384, nhead=8, num_layers=4, dim_ff=1024, dropout=0.1):
        super().__init__()
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head        = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

    def _sinusoidal(self, N, d_model, device):
        pe  = torch.zeros(N, d_model, device=device)
        pos = torch.arange(N, device=device).float().unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, device=device).float()
            * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe

    def forward(self, x):
        N  = x.shape[1]
        pe = self._sinusoidal(N, x.shape[-1], x.device).unsqueeze(0)
        x  = x + pe
        x  = self.transformer(x)
        return self.head(x).squeeze(-1).squeeze(0)   # [N]


# ── Load model ────────────────────────────────────────────────────────────────
model = TemporalTransformer().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Model loaded")

# ── Get test videos ───────────────────────────────────────────────────────────
test_videos = sorted(
    [f.replace(".pt", "") for f in os.listdir(TEST_FEAT_DIR)
     if f.endswith(".pt") and not f.startswith("dataset")],
    key=lambda x: int(x.split("_")[1])   # numeric sort: video_5611, video_5612...
)
print(f"Total test videos: {len(test_videos)}")

# ── Inference ─────────────────────────────────────────────────────────────────
rows = []

with torch.no_grad():
    for vid in test_videos:
        feat_path = os.path.join(TEST_FEAT_DIR, vid + ".pt")
        feats     = torch.load(feat_path, map_location=DEVICE)  # [N, 384]
        N         = len(feats)

        feats  = feats.unsqueeze(0)          # [1, N, 384]
        scores = model(feats).cpu().numpy()  # [N]

        # argsort ascending → frame with lowest score goes first
        # +1 for 1-indexed submission
        order = (np.argsort(scores) + 1).tolist()

        rows.append({"ID": vid, "order": order})

# ── Write CSV ─────────────────────────────────────────────────────────────────
output_path = "/kaggle/working/submission.csv"
with open(output_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["ID", "order"])
    writer.writeheader()
    for row in rows:
        order_str = str(row["order"])   # "[1, 2, 3, ...]" format
        f.write(f'{row["ID"]},"{order_str}"\n')

print(f"\nSubmission saved: {output_path}")
print(f"Total videos: {len(rows)}")

# ── Sanity check ──────────────────────────────────────────────────────────────
print("\nSanity check:")
all_ok = True
for row in rows:
    order = row["order"]
    N     = len(order)
    if min(order) != 1:
        print(f"  ERROR {row['ID']}: min={min(order)} expected 1")
        all_ok = False
    if max(order) != N:
        print(f"  ERROR {row['ID']}: max={max(order)} expected {N}")
        all_ok = False
    if len(set(order)) != N:
        print(f"  ERROR {row['ID']}: duplicates found")
        all_ok = False

if all_ok:
    print("  All videos passed — 1-indexed, no duplicates, correct length")

print("\nSample rows:")
for row in rows[:3]:
    print(f"  {row['ID']}: N={len(row['order'])}, min={min(row['order'])}, max={max(row['order'])}")
    print(f"  first 10: {row['order'][:10]}")

Model loaded
Total test videos: 296

Submission saved: /kaggle/working/submission.csv
Total videos: 296

Sanity check:
  All videos passed — 1-indexed, no duplicates, correct length

Sample rows:
  video_5611: N=100, min=1, max=100
  first 10: [2, 26, 11, 25, 5, 3, 16, 9, 4, 14]
  video_5612: N=122, min=1, max=122
  first 10: [122, 108, 121, 107, 120, 114, 113, 115, 109, 110]
  video_5613: N=97, min=1, max=97
  first 10: [49, 46, 47, 45, 19, 44, 48, 41, 37, 40]
